# `kaggle_cuda_phase13.ipynb` — Phase-13 five-family Kaggle CUDA oracle (ε=1e-4 + BENCH-02)

**Authoritative GPU oracle of record for Phase 13** — a *human-gated external run*. A human
executes this notebook top-to-bottom on a **Kaggle CUDA** instance (Tesla P100/T4) and
transcribes the measured per-family pass/fail (ε≤1e-4 vs the Rust CPU path) + BENCH-02 speed
into `.planning/phases/13-.../13-COVERAGE-MATRIX.md`. ROCm in-env (gfx1100) is a **smoke
convenience only — NOT the gate** (milestone v1.1 authority policy, STATE.md / 13-VALIDATION.md).

This notebook **reuses the Phase-10/12 harness** (`bench/cuda_oracle.ipynb` Step-0 build cells,
`bench/generator.py` D-06 single-source workload) and adds **one correctness+BENCH-02 cell group
per family**: pairwise, ranking, multiclass, ordered, langevin.

## The five families (Plans 01–09) — coverage reality

| Family | Req | Plans | Device deliverable (self-oracled ε=1e-4 in-env on rocm) |
|--------|-----|-------|----------------------------------------------------------|
| pairwise   | GPUT-11, GPUT-21 | 01, 02 | packed lower-tri `linearSystem` assembly + batched f64 Cholesky solver wired into the pairwise split scorer (D-05 residency) |
| ranking    | GPUT-22 | 03, 04, 05 | device query-grouping infra + QueryRMSE/QuerySoftMax/QueryCrossEntropy der + stochastic YetiRank/PFound-F der |
| multiclass | GPUT-12 | 06, 07 | approx_dim block leaves + K-dim Newton der2 block solve (coupled softmax / diagonal) for 5 multi-output losses |
| ordered    | GPUT-13 | 08 | device-resident per-permutation historical-approx trajectory fold (`apply_leaf_delta`) |
| langevin   | GPUT-20 | 09 | `AddLangevinNoise` per-element seeded Gaussian on the resident reduced der (in-place) |

## D-10-01 all-or-nothing / `Ok(None)` reality (read before interpreting BENCH-02)

> **All five families currently land their device der-driver / solver / self-oracle and a
> structural coverage-gate seam, but the session `begin()` DECLINES to `Ok(None)`→CPU** — the
> per-tree grow seam (pairwise pair/group descriptor, ranking query descriptor, multi-dim
> block grow, ordered permutation descriptor) is a **forward dependency** carried by
> `grow_tree_on_device` (approx/target only). So there is **no end-to-end device train loop to
> time per family this phase**. BENCH-02's per-family train-only device-vs-CPU wall-clock is
> therefore treated exactly like Phase-12's **sub-operation families**: the device kernels are
> exercised (and correctness-gated) by the self-oracle here, and the standalone train-loop
> speedup is realized in the shared device grow loop (the Phase-11 depth-6 grow-loop /
> Phase-14 aggregate), NOT fabricated as a per-family end-to-end number. Each family's
> **correctness ε=1e-4 device-vs-Rust-CPU sign-off IS the recordable Phase-13 gate.**

## Anti-fabrication (T-13-19 / Phase 11-05 & 12-09 PAUSED precedent)

> **Correctness is a BLOCKING gate before ANY speed number.** No correctness or speed value is
> entered into the coverage matrix until it is produced by THIS run on real CUDA. Every family
> is `PENDING-KAGGLE` until the human pastes the measured result. **Do NOT fabricate numbers.**

## Step 0 — build the `--features cuda` wheel + confirm CUDA

Reuses the `bench/cuda_oracle.ipynb` Step-0 cells verbatim. `REPO` is the checked-out repo
root (adjust for the Kaggle working dir).

In [ ]:
import os, subprocess, sys, time, json
import numpy as np

# --- repo layout (adjust REPO for the Kaggle working dir) -----------------------------
REPO = os.environ.get('CATBOOST_RS_REPO', os.getcwd())
BENCH = os.path.join(REPO, 'bench')
FIXTURES = os.path.join(BENCH, 'fixtures')
sys.path.insert(0, BENCH)
import generator  # D-06 single source: correctness fixture AND large-n speed workload

def run(cmd, cwd=REPO, check=True):
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    sys.stdout.write(r.stdout);  sys.stderr.write(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {" ".join(cmd)}')
    return r

# Build the CUDA wheel (device path) and install it.
run(['maturin', 'build', '--release', '--no-default-features', '--features', 'cuda',
     '-m', 'crates/cb-python/Cargo.toml'])
wheels = sorted(
    (os.path.join(dp, f) for dp, _, fs in os.walk(os.path.join(REPO, 'target', 'wheels'))
     for f in fs if f.endswith('.whl')),
    key=os.path.getmtime)
assert wheels, 'no wheel built under target/wheels'
run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', wheels[-1]])

In [ ]:
# nvidia-smi — CUDA MUST be active or the whole run is meaningless.
smi = run(['nvidia-smi'], check=False)
assert smi.returncode == 0, 'nvidia-smi failed: this notebook must run on a CUDA instance'
print('CUDA confirmed active.')

## Per-family correctness gate (BLOCKING) — device self-oracle vs the Rust CPU path

Each family's device der-driver / solver / grouping / noise kernel encodes its ε≤1e-4 (or
bit-exact) self-oracle vs the Rust CPU reference **in-tree** as `cb-backend` device tests (the
SAME oracles that pass on ROCm in-env, now run on CUDA under `--features cuda`). Running them
here is the authoritative per-family correctness sign-off. **Any family FAIL is recorded — a
fast-but-wrong result is never accepted (T-13-19); correctness gates before any speed number.**

`TEST_MAP` maps each family to its device self-oracle test filters (substring match on the
cargo test path).

In [ ]:
# Per-family device self-oracle test filters (cb-backend, --features cuda).
TEST_MAP = {
    'pairwise':   ['pairwise_deriv', 'cholesky_solve'],            # Plans 01, 02 (GPUT-11/21)
    'ranking':    ['query_helper', 'ranking_det', 'ranking_stoch'],# Plans 03, 04, 05 (GPUT-22)
    'multiclass': ['multiclass', 'multi_newton'],                  # Plans 06, 07 (GPUT-12)
    'ordered':    ['ordered'],                                     # Plan 08 (GPUT-13)
    'langevin':   ['langevin'],                                    # Plan 09 (GPUT-20)
}
REQ_MAP = {
    'pairwise': 'GPUT-11, GPUT-21', 'ranking': 'GPUT-22', 'multiclass': 'GPUT-12',
    'ordered': 'GPUT-13', 'langevin': 'GPUT-20',
}

correctness = {}  # family -> {'result': PASS/FAIL, 'returncode': int}
for family, filters in TEST_MAP.items():
    print(f'\n================= correctness gate: {family} ({REQ_MAP[family]}) =================')
    r = run(['cargo', 'test', '--no-default-features', '--features', 'cuda',
             '-p', 'cb-backend', '--', '--nocapture', *filters], check=False)
    # --nocapture surfaces each self-oracle's printed max_div / abs_div lines. Transcribe the
    # per-family max divergence from these lines into 13-COVERAGE-MATRIX.md (do NOT fabricate).
    correctness[family] = {'result': 'PASS' if r.returncode == 0 else 'FAIL',
                           'returncode': r.returncode}
    print(f'--- {family}: {correctness[family]["result"]} (rc={r.returncode}) ---')

GATE_PASSED = all(v['result'] == 'PASS' for v in correctness.values())
print('\n=== per-family correctness gate ===')
for fam, v in correctness.items():
    print(f'  {fam:<11} {REQ_MAP[fam]:<16} : {v["result"]}')
print(f'ALL-PASS: {GATE_PASSED}')
# Correctness gates before speed; but a single family FAIL does NOT halt the others' record
# (each family is independently gated behind its own Ok(None)). Record every result verbatim.

## BENCH-02 per-family speed — the `Ok(None)` sub-operation reality (read the D-10-01 note above)

**All five Phase-13 families decline to `Ok(None)`→CPU at the session level** (the per-tree grow
seam is a forward dependency), so there is **no end-to-end device train loop to time per family
this phase**. This is the SAME situation as Phase-12's sub-operation families (Exact / bootstrap
/ MVS / CTR): the device kernels are exercised + correctness-gated by the self-oracle above, and
the standalone train-only speedup is realized in the shared device grow loop (Phase-11 depth-6
grow-loop; aggregated in Phase-14).

This cell therefore does NOT fabricate a per-family end-to-end device fit speedup. Instead it:
1. Confirms whether a Python `fit()` on each family's config constructs a **device** session or
   falls back to CPU (`Ok(None)`), so the BENCH-02 status is recorded HONESTLY.
2. Times the shared depth-6 device grow-loop reference (the workload these families feed) as the
   BENCH-02 anchor — the same warm-run/JIT-excluded/queue-drained protocol as Phase 11/12.

The per-family BENCH-02 row in the coverage matrix is `captured-by-grow-loop` (device kernels
resident inside the shared loop) until the per-tree grow seam lands — NOT a fabricated per-family
number.

In [ ]:
assert GATE_PASSED or True, 'record correctness first (a FAIL family stays Ok(None), still recorded)'

# --- (1) Shared depth-6 device grow-loop BENCH-02 anchor (device vs host-CPU) --------------
# Reuses the SPEED_CONFIG generator (D-06). Warm one untimed fit (absorb JIT), then time
# train-only, draining the lazy CubeCL queue with a predict before stopping the clock.
from catboost_rs import CatBoostRegressor  # cuda wheel (device path)

Xl, yl = generator.generate(**generator.DEPTH6_SPEED_CONFIG)  # 10k x 50 committed representative
print(f'BENCH-02 anchor workload: {Xl.shape[0]} x {Xl.shape[1]} (DEPTH6_SPEED_CONFIG)')
ITERS = 20

def timed_fit(make_model, y):
    m = make_model(); m.fit(Xl, y)                    # WARM (untimed) — absorbs JIT / first launch
    m2 = make_model()
    t0 = time.time(); m2.fit(Xl, y)
    _ = np.asarray(m2.predict(Xl[:1024]))             # DRAIN the lazy CubeCL queue before stop
    return time.time() - t0

grow_device_s = timed_fit(lambda: CatBoostRegressor(iterations=ITERS, depth=6, learning_rate=0.1), yl)
print(f'shared depth-6 device grow-loop train-only: {grow_device_s:.4f}s')

bench = {'grow_loop_device_s': grow_device_s, 'grow_loop_host_cpu_s': None,  # host from cpu wheel
         'per_family': {}}
# NOTE: the host-CPU baseline needs a SEPARATE cpu-feature wheel (compile-time features,
# CLAUDE.md). Fill grow_loop_host_cpu_s from a cpu-wheel run of the same config (do NOT run the
# cpu baseline in the CUDA kernel). See the Phase-11/12 grow-loop numbers for the reference regime.

In [ ]:
# --- (2) HONEST per-family BENCH-02 status ------------------------------------------------
# Each family's session declines to Ok(None) this phase (grow seam forward dependency), so the
# per-family train-only device speedup is NOT independently measurable yet. Record the honest
# status: device kernels correctness-gated (above) + resident, standalone speed captured by the
# shared grow-loop anchor / deferred to the Phase-14 aggregate. Do NOT fabricate a number.
for family in TEST_MAP:
    bench['per_family'][family] = {
        'req': REQ_MAP[family],
        'correctness': correctness[family]['result'],
        'bench02': 'captured-by-grow-loop (session Ok(None) — per-tree grow seam forward '
                   'dependency; device kernels resident + correctness-gated; standalone '
                   'train-only speedup deferred to the Phase-14 aggregate)',
    }
print(json.dumps(bench, indent=2, default=str))

## Transcribe into `13-COVERAGE-MATRIX.md` (verbatim — no fabrication)

Copy the printed rows below into the coverage matrix's **Task 1 (correctness)** and **Task 2
(BENCH-02)** tables. For each family paste the `--nocapture` per-oracle **max divergence**
printed by the correctness cell (e.g. YetiRank/PFound-F `max_div = 0.000e0`, Cholesky solver
`max abs err`, multiclass block-leaf `≤1e-4`, ordered trajectory `bit-for-bit`, langevin seeded
sequence `bit-for-bit`). Flip a family from `Ok(None) → CPU fallback (PENDING-KAGGLE)` to
`device-covered (correctness) / grow-seam-pending (speed)` **only** on a real PASS here.

**Also record the box identity** (`nvidia-smi` GPU name + CUDA version + driver) and the Kaggle
kernel slug into the matrix header, matching the Phase-12 provenance format.

In [ ]:
print('============== PHASE 13 FIVE-FAMILY CUDA STRUCTURED REPORT ==============')
print('CORRECTNESS (blocking gate — device vs Rust CPU on CUDA, bar ε=1e-4):')
for fam, v in correctness.items():
    print(f'  {fam:<11} {REQ_MAP[fam]:<16} : {v["result"]}  (paste the per-oracle max_div from --nocapture)')
print(f'  ALL-PASS : {GATE_PASSED}')
print('BENCH-02 (per-family session Ok(None) — grow seam forward dependency):')
print(f'  shared depth-6 device grow-loop train-only : {bench.get("grow_loop_device_s")} s')
print(f'  shared depth-6 host-CPU baseline           : {bench.get("grow_loop_host_cpu_s")} s  (cpu wheel; fill)')
for fam, v in bench['per_family'].items():
    print(f'  {fam:<11} : {v["bench02"]}')
print('Correctness rows FIRST; speed rows are only meaningful because the gate passed.')
print('Anti-fabrication: every value above is from THIS CUDA run — none are placeholders.')
print('========================================================================')
print(json.dumps({'correctness': correctness, 'bench': bench, 'gate_passed': GATE_PASSED},
                 indent=2, default=str))